# 1. Introduction to Libraries

In this notebook, we will build a CNN‐LSTM hybrid model in PyTorch to predict hourly energy consumption in the ComEd region (Illinois), using weather data from multiple stations.  
Below, we list and briefly explain each library we use.

- **numpy**: Fundamental package for array computing.
- **pandas**: DataFrame structures and I/O (reading CSV, handling dates).
- **matplotlib** & **seaborn**: Plotting and visualization.
- **scikit‐learn**: Data splitting, metrics (RMSE, MAPE), and TimeSeriesSplit.
- **torch** (PyTorch): Model definition, tensors, training loops.
- **torch.nn** and **torch.optim**: Neural network layers, loss functions, optimizers.
- **tqdm**: Progress bars.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# PyTorch imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Scikit-learn imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error


# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# 2. Data Loading & Preprocessing

Load the Data and preprocess it.

In [2]:
weather_df = pd.read_csv('weather_final.csv', sep=";")

In [3]:
weather_df.shape

(66505, 95)

In [4]:
weather_df.head(5)

,time,Unnamed: 0,temp_AUR,temp_BC,temp_DKC,temp_FRE,temp_KAN,temp_LYN,temp_MDW,temp_ORD,...,prcp_ORD_prcp_WOG_merged,prcp_RFD_prcp_ROC_merged,wdir_JOL_wdir_JRA_merged,wdir_SRA_wdir_SWD_merged,wspd_JOL_wspd_JRA_merged,wspd_SRA_wspd_SWD_merged,pres_AUR_pres_DKC_merged,pres_JOL_pres_LYN_merged,pres_RFD_pres_ROC_merged,pres_SRA_pres_SWD_merged
0,2011-01-01 00:00:00,0.0,11.283934,9.959076,10.862553,10.263194,12.604149,11.72351,11.1,11.1,...,1.449894,0.993657,188.256626,170.0,25.602067,24.1,1005.381517,1005.287782,1003.9,1005.2
1,2011-01-01 01:00:00,1.0,11.700000,10.000000,11.700000,11.245949,13.479755,12.80000,12.8,11.7,...,0.000000,0.000000,170.000000,170.0,24.100000,31.7,1005.600000,1005.600000,1005.2,1005.1
2,2011-01-01 02:00:00,2.0,11.700000,12.200000,11.700000,10.532717,13.211125,12.80000,12.8,11.7,...,0.000000,0.012270,190.000000,210.0,14.800000,20.5,1005.300000,1005.600000,1004.7,1005.1
3,2011-01-01 03:00:00,3.0,10.600000,12.200000,10.600000,8.564046,14.558970,14.00000,12.2,11.1,...,0.000000,0.500000,170.000000,160.0,14.800000,22.3,1005.000000,1005.500000,1004.1,1004.9
4,2011-01-01 04:00:00,4.0,11.100000,12.200000,11.100000,3.215452,12.999388,12.80000,12.8,11.1,...,0.000000,2.500000,170.000000,180.0,20.500000,24.1,1005.100000,1005.300000,1006.7,1004.6


In [5]:
weather_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66505 entries, 0 to 66504
Data columns (total 95 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   time                      66505 non-null  object 
 1   Unnamed: 0                66505 non-null  float64
 2   temp_AUR                  66505 non-null  float64
 3   temp_BC                   66505 non-null  float64
 4   temp_DKC                  66505 non-null  float64
 5   temp_FRE                  66505 non-null  float64
 6   temp_KAN                  66505 non-null  float64
 7   temp_LYN                  66505 non-null  float64
 8   temp_MDW                  66505 non-null  float64
 9   temp_ORD                  66505 non-null  float64
 10  temp_PON                  66505 non-null  float64
 11  temp_RFD                  66505 non-null  float64
 12  temp_ROC                  66505 non-null  float64
 13  temp_SAR                  66505 non-null  float64
 14  temp_S

In [6]:
weather_df.describe()

,Unnamed: 0,temp_AUR,temp_BC,temp_DKC,temp_FRE,temp_KAN,temp_LYN,temp_MDW,temp_ORD,temp_PON,...,prcp_ORD_prcp_WOG_merged,prcp_RFD_prcp_ROC_merged,wdir_JOL_wdir_JRA_merged,wdir_SRA_wdir_SWD_merged,wspd_JOL_wspd_JRA_merged,wspd_SRA_wspd_SWD_merged,pres_AUR_pres_DKC_merged,pres_JOL_pres_LYN_merged,pres_RFD_pres_ROC_merged,pres_SRA_pres_SWD_merged
count,66505.000000,66505.000000,66505.000000,66505.000000,66505.000000,66505.000000,66505.000000,66505.000000,66505.000000,66505.000000,...,66505.000000,66505.000000,66505.000000,66505.000000,66505.000000,66505.000000,66505.000000,66505.000000,66505.000000,66505.000000
mean,33252.000000,9.987097,9.183546,9.473234,9.113604,10.781685,10.341004,11.435379,10.642564,11.425053,...,0.110749,0.139137,195.092889,194.296528,14.814750,14.384708,1016.795043,1016.611778,1017.110059,1016.498249
std,19198.484163,11.887069,11.255387,11.763934,11.775790,11.547228,11.329474,11.560850,11.599123,11.674985,...,0.700252,0.852101,96.513890,98.656244,9.096019,9.005014,7.540886,7.704776,7.638775,7.506906
min,0.000000,-30.000000,-26.700000,-28.300000,-27.300000,-29.300000,-26.200000,-25.600000,-26.700000,-29.500000,...,0.000000,-0.379252,10.000000,7.000000,0.000000,0.000000,982.900000,983.200000,984.700000,982.600000
25%,16626.000000,1.100000,1.100000,0.500000,0.300000,1.700000,1.500000,2.200000,1.700000,2.000000,...,0.000000,0.000000,120.000000,120.000000,9.400000,7.600000,1012.100000,1011.700000,1012.300000,1011.900000
50%,33252.000000,10.600000,9.400000,10.100000,9.600000,11.400000,10.600000,11.700000,11.100000,12.000000,...,0.000000,0.000000,200.000000,200.000000,13.000000,13.000000,1016.500000,1016.200000,1016.800000,1016.300000
75%,49878.000000,20.000000,18.300000,19.200000,18.600000,20.200000,19.800000,21.100000,20.600000,21.000000,...,0.000000,0.000000,270.000000,275.000000,20.500000,20.500000,1021.500000,1021.400000,1021.900000,1021.200000
max,66504.000000,38.300000,38.900000,37.600000,36.700000,38.000000,39.000000,39.400000,39.400000,39.000000,...,38.850000,36.800000,370.043122,360.000000,70.600000,61.200000,1044.800000,1044.800000,1045.200000,1043.900000


In [7]:
# Convert the "time" column into a true datetime type
weather_df["time"] = pd.to_datetime(weather_df["time"], format="%Y-%m-%d %H:%M:%S")

# Set "time" as the index and sort by it (just in case)
weather_df = weather_df.set_index("time").sort_index()

# Verify the changes
print(weather_df.index.min(), weather_df.index.max())
print(weather_df.index.freq)

2011-01-01 00:00:00 2018-08-03 00:00:00
None


In [8]:
# Feature Engineering: Extract hour‐of‐day, day‐of‐week, and month as new columns
weather_df["hour"] = weather_df.index.hour
weather_df["day_of_week"] = weather_df.index.dayofweek # we are starting from 0, 0 -Monday, 6-Sunday
weather_df["month"] = weather_df.index.month

# “weekend vs. weekday” as a boolean:
weather_df["is_weekend"] = weather_df["day_of_week"].isin([5, 6]).astype(int)

# View the first few rows to confirm
weather_df[["hour", "day_of_week", "month", "is_weekend"]].head(5)


,hour,day_of_week,month,is_weekend
time,,,,
2011-01-01 00:00:00,0,5,1,1
2011-01-01 01:00:00,1,5,1,1
2011-01-01 02:00:00,2,5,1,1
2011-01-01 03:00:00,3,5,1,1
2011-01-01 04:00:00,4,5,1,1


In [9]:
# Dropping the now‐redundant "Unnamed: 0" column
weather_df = weather_df.drop(columns=["Unnamed: 0"])
weather_df.head(5)

,temp_AUR,temp_BC,temp_DKC,temp_FRE,temp_KAN,temp_LYN,temp_MDW,temp_ORD,temp_PON,temp_RFD,...,wspd_JOL_wspd_JRA_merged,wspd_SRA_wspd_SWD_merged,pres_AUR_pres_DKC_merged,pres_JOL_pres_LYN_merged,pres_RFD_pres_ROC_merged,pres_SRA_pres_SWD_merged,hour,day_of_week,month,is_weekend
time,,,,,,,,,,,,,,,,,,,,,
2011-01-01 00:00:00,11.283934,9.959076,10.862553,10.263194,12.604149,11.72351,11.1,11.1,12.250059,10.0,...,25.602067,24.1,1005.381517,1005.287782,1003.9,1005.2,0,5,1,1
2011-01-01 01:00:00,11.700000,10.000000,11.700000,11.245949,13.479755,12.80000,12.8,11.7,13.848533,11.7,...,24.100000,31.7,1005.600000,1005.600000,1005.2,1005.1,1,5,1,1
2011-01-01 02:00:00,11.700000,12.200000,11.700000,10.532717,13.211125,12.80000,12.8,11.7,13.243095,11.1,...,14.800000,20.5,1005.300000,1005.600000,1004.7,1005.1,2,5,1,1
2011-01-01 03:00:00,10.600000,12.200000,10.600000,8.564046,14.558970,14.00000,12.2,11.1,14.303224,11.1,...,14.800000,22.3,1005.000000,1005.500000,1004.1,1004.9,3,5,1,1
2011-01-01 04:00:00,11.100000,12.200000,11.100000,3.215452,12.999388,12.80000,12.8,11.1,13.210418,4.4,...,20.500000,24.1,1005.100000,1005.300000,1006.7,1004.6,4,5,1,1


In [10]:
TEMP = ['temp_AUR', 'temp_BC', 'temp_DKC', 'temp_FRE', 'temp_KAN', 'temp_LYN', 'temp_MDW', 'temp_ORD', 'temp_PON', 'temp_RFD', 'temp_ROC', 'temp_SAR',                  
   'temp_SRF', 'temp_WOG']

In [11]:
 # Feature selection (use all features except 'Temperature' as X; assume target column is 'TEMP' or similar)
target_col = 'TEMP'
feature_cols = [col for col in weather_df.columns if col != target_col]

X = weather_df[feature_cols].values
y = weather_df[target_col].values

KeyError: 'TEMP'

In [ ]:
# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# Chronological split: Use first 80% for training, last 20% for testing
split_idx = int(0.7 * len(df))
X_train, X_test = X_scaled[:split_idx], X_scaled[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]